# Author formulas and inspect their mathematics

Run these cells in order using the production environment with
`pip install -e '.[notebook]'` (from `production/`). No payroll data or
analysis run is needed. Edit an expression and rerun its preview cell.

Python identifiers remain descriptive. Display symbols are notebook settings;
they do not change the model or its fingerprint.


In [1]:
from labor_cost import (
    Factor,
    FieldSpec,
    Game,
    ModelGraph,
    ModelSpec,
    Notation,
    RowCost,
    SourceSchema,
    Switch,
    aligned_mean,
    child,
    field,
    local_game,
    organization_game,
    outer_factors,
    render_formula,
    render_model,
    sum_over,
    value,
)

row_pay = field("rate") * field("hours") + field("allowance")
render_formula(row_pay)


### Formula

$$
\mathrm{RAT} \cdot \mathrm{HOU} + \mathrm{ALL}
$$

Expression preview; this does not validate a model.

Structural preview: dimensions, units and amount bases are unspecified without model context.

### Symbol legend

Symbol | Kind / identifier | Source | Population / scope | Dimensions | Units / basis
--- | --- | --- | --- | --- | ---
$\mathrm{ALL}$ | field: allowance | unspecified | unspecified / unspecified | unspecified | unspecified / unspecified
$\mathrm{HOU}$ | field: hours | unspecified | unspecified / unspecified | unspecified | unspecified / unspecified
$\mathrm{RAT}$ | field: rate | unspecified | unspecified / unspecified | unspecified | unspecified / unspecified

## Choose display symbols

Defaults use initials or the first three letters. Collisions receive
qualifiers. Override any symbol; longer names are allowed. Explicit symbols
must be unique across namespaces. Reuse this configuration for all previews.


In [2]:
notation = Notation(
    fields={"rate": "RT", "hours": "H", "allowance": "ALW"},
    factors={
        "rates": "R", "hours": "T", "members": "M",
        "first_mix": "PG", "conditional_mix": "PR", "allowances": "A",
    },
    games={"organization": "LC", "local": "TC", "unit": "UC",
           "earnings": "BC"},
    dimensions={"grade": "g", "region": "r", "team": "t"},
)
render_formula(row_pay, notation=notation)


### Formula

$$
\mathrm{RT} \cdot \mathrm{H} + \mathrm{ALW}
$$

Expression preview; this does not validate a model.

Structural preview: dimensions, units and amount bases are unspecified without model context.

### Symbol legend

Symbol | Kind / identifier | Source | Population / scope | Dimensions | Units / basis
--- | --- | --- | --- | --- | ---
$\mathrm{ALW}$ | field: allowance | unspecified | unspecified / unspecified | unspecified | unspecified / unspecified
$\mathrm{H}$ | field: hours | unspecified | unspecified / unspecified | unspecified | unspecified / unspecified
$\mathrm{RT}$ | field: rate | unspecified | unspecified / unspecified | unspecified | unspecified / unspecified

## Declare the workforce representation

This model uses P(grade) × P(region | grade), paired hourly rate × hours,
and a separate population-average allowance. Membership carries the weights
and exposure conversions for the complete paired product. The following
structural preview works before we have a complete model.


In [3]:
paired = aligned_mean(value("rates") * value("hours"), "members")
joint = value("first_mix") * value("conditional_mix")
earnings_formula = sum_over(joint * paired, "grade", "region")
render_formula(earnings_formula, notation=notation)


### Formula

$$
\sum_{\mathrm{g},\mathrm{r}}\left(\mathrm{PG} \cdot \mathrm{PR} \cdot \operatorname{AlignedMean}_{\mathrm{M}}\!\left(\mathrm{R} \cdot \mathrm{T}\right)\right)
$$

Expression preview; this does not validate a model.

Structural preview: dimensions, units and amount bases are unspecified without model context.

### Symbol legend

Symbol | Kind / identifier | Source | Population / scope | Dimensions | Units / basis
--- | --- | --- | --- | --- | ---
$\mathrm{g}$ | dimension: grade | unspecified | unspecified / unspecified | unspecified | unspecified / unspecified
$\mathrm{r}$ | dimension: region | unspecified | unspecified / unspecified | unspecified | unspecified / unspecified
$\mathrm{PR}$ | factor: conditional&#95;mix | unspecified | unspecified / unspecified | unspecified | unspecified / unspecified
$\mathrm{PG}$ | factor: first&#95;mix | unspecified | unspecified / unspecified | unspecified | unspecified / unspecified
$\mathrm{T}$ | factor: hours | unspecified | unspecified / unspecified | unspecified | unspecified / unspecified
$\mathrm{M}$ | factor: members | unspecified | unspecified / unspecified | unspecified | unspecified / unspecified
$\mathrm{R}$ | factor: rates | unspecified | unspecified / unspecified | unspecified | unspecified / unspecified

In [4]:
source = SourceSchema(dimensions={"grade": "grade", "region": "region"})
fields = (
    FieldSpec("rate", "hourly_rate", units="currency/time",
              constraint="positive"),
    FieldSpec("hours", "paid_hours", units="time", constraint="nonnegative"),
    FieldSpec("allowance", "allowance"),
)
factors = outer_factors() + (
    Factor("first_mix", "distribution", dimensions=("grade",)),
    Factor("conditional_mix", "distribution", dimensions=("region",),
           given=("grade",), fallbacks=((),)),
    Factor("members", "membership", dimensions=("grade", "region"),
           fallbacks=(("grade",), ("region",), ())),
    Factor("rates", "aligned", source="rate"),
    Factor("hours", "aligned", source="hours"),
    Factor("allowances", "payment", formula=field("allowance")),
)
games = (
    Game("earnings", earnings_formula, (
        Switch("first_mix"), Switch("conditional_mix"), Switch("members"),
        Switch("earned_pay_inputs", ("rates", "hours")),
    )),
    Game("unit", child("earnings") + value("allowances"), (
        Switch("base_earnings", child="earnings"), Switch("allowances"),
    )),
)
model = ModelSpec("hourly-preview", "1.0.0", graph=ModelGraph(
    source=source, populations=(), fields=fields,
    row_costs=(RowCost("all", row_pay),), factors=factors, games=games,
    team_game="unit", organization=organization_game("unit"),
    local=local_game("unit"),
))
render_formula(earnings_formula, model=model, notation=notation)


### Formula

$$
\sum_{\mathrm{g},\mathrm{r}}\left(\begin{aligned}&\mathrm{PG}\!\left(\mathrm{g};\mathrm{t}\right) \cdot \mathrm{PR}\!\left(\mathrm{r}\mid \mathrm{g};\mathrm{t}\right) \\ &{}\cdot \sum_{i}\left(w^{(\mathrm{M})}_{i\mid \mathrm{t},\mathrm{g},\mathrm{r}} \cdot a^{(\mathrm{M})}_{i\mid \mathrm{t},\mathrm{g},\mathrm{r}} \cdot \left(\mathrm{R}_{i} \cdot \mathrm{T}_{i}\right)\right)\end{aligned}\right)
$$

Expression preview; this does not validate a model.

i identifies an aligned employee/employment-type record. Each w is a normalized sparse membership weight. The matching a belongs to that same membership object and switches with it.

members: actual basis; a = inverse donor-row exposure.

### Symbol legend

Symbol | Kind / identifier | Source | Population / scope | Dimensions | Units / basis
--- | --- | --- | --- | --- | ---
$\mathrm{g}$ | dimension: grade | grade | unspecified / unspecified | unspecified | category / unspecified
$\mathrm{r}$ | dimension: region | region | unspecified / unspecified | unspecified | category / unspecified
$\mathrm{t}$ | dimension: team | team | unspecified / unspecified | unspecified | category / unspecified
$\mathrm{PR}$ | distribution: conditional&#95;mix | unspecified | all / team | grade, region | dimensionless / unspecified
$\mathrm{PG}$ | distribution: first&#95;mix | unspecified | all / team | grade | dimensionless / unspecified
$\mathrm{T}$ | aligned: hours | paid&#95;hours | all / team | employee/employment&#95;type | time / unspecified
$\mathrm{M}$ | membership: members | unspecified | all / team | grade, region | weights: dimensionless; conversion: 1/exposure / actual
$\mathrm{R}$ | aligned: rates | hourly&#95;rate | all / team | employee/employment&#95;type | currency/time / unspecified

## Inspect the complete equation sheet

Compilation checks declarations without running attribution. It does not
establish business correctness or reconciliation against source data. The
sheet keeps row costs, organizational costs, local costs, and nested games
separate. Switches describe whole-object interventions, not extra factors.


In [5]:
preview = render_model(model, notation=notation)
preview


### Row cost 1: all

$$
\operatorname{RowCost}^{(1)}_{\mathrm{ALL}} = \mathrm{RT} \cdot \mathrm{H} + \mathrm{ALW}
$$

Actual employee/employment-type/month amount; currency. Overlapping applicable row components add.

### Cost: organization

$$
\mathrm{LC} = \mathrm{WS} \cdot \sum_{\mathrm{t}}\left(\mathrm{TM}\!\left(\mathrm{t}\right) \cdot \mathrm{UC}_{\mathrm{t}}\right)
$$

Scope: organization; population: all. Switches: workforce&#95;scale &#40;Workforce scale&#41; -&gt; workforce&#95;scale &#40;whole objects&#41;; team&#95;mix &#40;Team mix&#41; -&gt; team&#95;mix &#40;whole objects&#41;; team&#95;economics &#40;Team economics&#41; -&gt; child unit

### Cost: local

$$
\mathrm{TC} = \sum_{\mathrm{t}}\left(\mathrm{TE}_{\mathrm{t}} \cdot \mathrm{UC}_{\mathrm{t}}\right)
$$

Scope: local team; population: all. Team-axis sums use the selected local team only. Switches: workforce&#95;scale &#40;Workforce scale&#41; -&gt; team&#95;exposure &#40;whole objects&#41;; team&#95;economics &#40;Team economics&#41; -&gt; child unit

### Cost: earnings

$$
\mathrm{BC}_{\mathrm{t}} = \sum_{\mathrm{g},\mathrm{r}}\left(\begin{aligned}&\mathrm{PG}\!\left(\mathrm{g};\mathrm{t}\right) \cdot \mathrm{PR}\!\left(\mathrm{r}\mid \mathrm{g};\mathrm{t}\right) \\ &{}\cdot \sum_{i}\left(w^{(\mathrm{M})}_{i\mid \mathrm{t},\mathrm{g},\mathrm{r}} \cdot a^{(\mathrm{M})}_{i\mid \mathrm{t},\mathrm{g},\mathrm{r}} \cdot \left(\mathrm{R}_{i} \cdot \mathrm{T}_{i}\right)\right)\end{aligned}\right)
$$

Scope: team; population: all. Switches: first&#95;mix -&gt; first&#95;mix &#40;whole objects&#41;; conditional&#95;mix -&gt; conditional&#95;mix &#40;whole objects&#41;; members -&gt; members &#40;whole objects&#41;; earned&#95;pay&#95;inputs -&gt; rates, hours &#40;whole objects&#41;

### Cost: unit

$$
\mathrm{UC}_{\mathrm{t}} = \mathrm{BC}_{\mathrm{t}} + \mathrm{A}_{\mathrm{t}}
$$

Scope: team; population: all. Switches: base&#95;earnings -&gt; child earnings; allowances -&gt; allowances &#40;whole objects&#41;

Compilation validates declarations, not business correctness or data reconciliation. Organizational and local costs are separate views; do not add their allocations.

i identifies an aligned employee/employment-type record. Each w is a normalized sparse membership weight. The matching a belongs to that same membership object and switches with it.

members: actual basis; a = inverse donor-row exposure.

### Symbol legend

Symbol | Kind / identifier | Source | Population / scope | Dimensions | Units / basis
--- | --- | --- | --- | --- | ---
$\mathrm{g}$ | dimension: grade | grade | unspecified / unspecified | unspecified | category / unspecified
$\mathrm{r}$ | dimension: region | region | unspecified / unspecified | unspecified | category / unspecified
$\mathrm{t}$ | dimension: team | team | unspecified / unspecified | unspecified | category / unspecified
$\mathrm{A}$ | payment: allowances | allowance | all / team | scalar | currency/exposure / actual
$\mathrm{PR}$ | distribution: conditional&#95;mix | unspecified | all / team | grade, region | dimensionless / unspecified
$\mathrm{PG}$ | distribution: first&#95;mix | unspecified | all / team | grade | dimensionless / unspecified
$\mathrm{T}$ | aligned: hours | paid&#95;hours | all / team | employee/employment&#95;type | time / unspecified
$\mathrm{M}$ | membership: members | unspecified | all / team | grade, region | weights: dimensionless; conversion: 1/exposure / actual
$\mathrm{R}$ | aligned: rates | hourly&#95;rate | all / team | employee/employment&#95;type | currency/time / unspecified
$\mathrm{TE}$ | total: team&#95;exposure | unspecified | all / organization | team | exposure / unspecified
$\mathrm{TM}$ | distribution: team&#95;mix | unspecified | all / organization | team | dimensionless / unspecified
$\mathrm{WS}$ | total: workforce&#95;scale | unspecified | all / organization | scalar | exposure / unspecified
$\mathrm{ALW}$ | field: allowance | allowance | all / unspecified | unspecified | currency / unspecified
$\mathrm{H}$ | field: hours | paid&#95;hours | all / unspecified | unspecified | time / unspecified
$\mathrm{RT}$ | field: rate | hourly&#95;rate | all / unspecified | unspecified | currency/time / unspecified
$\mathrm{BC}$ | game: earnings | unspecified | all / team | unspecified | currency/exposure / unspecified
$\mathrm{TC}$ | game: local | unspecified | all / local team | unspecified | currency / unspecified
$\mathrm{LC}$ | game: organization | unspecified | all / organization | unspecified | currency / unspecified
$\mathrm{UC}$ | game: unit | unspecified | all / team | unspecified | currency/exposure / unspecified
$\mathrm{ALL}$ | population: all | unspecified | unspecified / unspecified | unspecified | unspecified / unspecified

## Expand one child when useful

The default equation sheet names child costs to keep large models readable.
Explicit expansion substitutes the child's authored expression. Full-model
context keeps symbols consistent between this preview and the equation sheet.


In [6]:
render_formula(
    games[1].formula, model=model, notation=notation, expand_children=True,
)


### Formula

$$
\begin{aligned}&\left(\sum_{\mathrm{g},\mathrm{r}}\left(\begin{aligned}&\mathrm{PG}\!\left(\mathrm{g};\mathrm{t}\right) \cdot \mathrm{PR}\!\left(\mathrm{r}\mid \mathrm{g};\mathrm{t}\right) \\ &{}\cdot \sum_{i}\left(w^{(\mathrm{M})}_{i\mid \mathrm{t},\mathrm{g},\mathrm{r}} \cdot a^{(\mathrm{M})}_{i\mid \mathrm{t},\mathrm{g},\mathrm{r}} \cdot \left(\mathrm{R}_{i} \cdot \mathrm{T}_{i}\right)\right)\end{aligned}\right)\right) \\ &{} + \mathrm{A}_{\mathrm{t}}\end{aligned}
$$

Expression preview; this does not validate a model.

i identifies an aligned employee/employment-type record. Each w is a normalized sparse membership weight. The matching a belongs to that same membership object and switches with it.

members: actual basis; a = inverse donor-row exposure.

### Symbol legend

Symbol | Kind / identifier | Source | Population / scope | Dimensions | Units / basis
--- | --- | --- | --- | --- | ---
$\mathrm{g}$ | dimension: grade | grade | unspecified / unspecified | unspecified | category / unspecified
$\mathrm{r}$ | dimension: region | region | unspecified / unspecified | unspecified | category / unspecified
$\mathrm{t}$ | dimension: team | team | unspecified / unspecified | unspecified | category / unspecified
$\mathrm{A}$ | payment: allowances | allowance | all / team | scalar | currency/exposure / actual
$\mathrm{PR}$ | distribution: conditional&#95;mix | unspecified | all / team | grade, region | dimensionless / unspecified
$\mathrm{PG}$ | distribution: first&#95;mix | unspecified | all / team | grade | dimensionless / unspecified
$\mathrm{T}$ | aligned: hours | paid&#95;hours | all / team | employee/employment&#95;type | time / unspecified
$\mathrm{M}$ | membership: members | unspecified | all / team | grade, region | weights: dimensionless; conversion: 1/exposure / actual
$\mathrm{R}$ | aligned: rates | hourly&#95;rate | all / team | employee/employment&#95;type | currency/time / unspecified
$\mathrm{BC}$ | game: earnings | unspecified | all / team | unspecified | currency/exposure / unspecified

## Reuse the output

`preview.latex` contains delimiter-free mathematics, `preview.markdown`
contains the complete sheet, and `preview.legend` contains structured symbol
metadata. For example, inspect the mapping for aligned rates below. None of
these presentation settings are stored in model publication or run bundles.


In [7]:
dict(next(item for item in preview.legend
          if item["namespace"] == "factors" and item["identifier"] == "rates"))


{'namespace': 'factors',
 'identifier': 'rates',
 'kind': 'aligned',
 'symbol': 'R',
 'latex': '\\mathrm{R}',
 'source': 'hourly_rate',
 'population': 'all',
 'dimensions': ('employee/employment_type',),
 'units': 'currency/time',
 'basis': None,
 'scope': 'team'}